In [ ]:
##Function to Join Tables --
def join_gens(gen = list):
    joined_tables = spark.sql(f"""
    SELECT *
    FROM df12mdp.brookefoye.gen{gen[0]}""")
    gen.remove(gen[0])
    #if you actually have to join gens
    while len(gen)>0:
        df = spark.sql(f"""SELECT * FROM df12mdp.brookefoye.gen{gen[0]}""")
        joined_tables = joined_tables.union(df)
        gen.remove(gen[0])
    joined_tables.write.mode("overwrite").saveAsTable("df12mdp.brookefoye.poke_team")
    return joined_tables

In [ ]:
##Random Number Generator -- FULLY TESTED
import random
#generates a random number between 1 and the len() of team_source table -1
def random_number_generator(minimum, maximum):
    return random.randint(minimum, maximum)

In [ ]:
##Fill in Blank Spaces for Types of Pokemon --FULLY TESTED
def type_checker(poke_type):
    if len(poke_type)<6:
        while len(poke_type)<6:
            type = random_number_generator(0, len(poke_type)-1)
            poke_type.append(poke_type[type].lower())
    return poke_type

In [ ]:
##Function to Build Team
import random

def build_team(team_source, poke_type):
    # Step 1: Filter Spark DataFrame by types
    filtered_df = team_source.filter(team_source["Type1"].isin(poke_type))
    
    # Step 2: Collect to Python
    rows = filtered_df.select("Name", "Type1").collect()
    
    if not rows:
        return {}

    # Step 3: Build pool of names (no duplicates)
    all_names = list({row["Name"] for row in rows})

    # Step 4: Pick up to 6 random unique names
    chosen_names = random.sample(all_names, min(6, len(all_names)))

    # Step 5: Return as dict
    return {f"slot{i+1}": name for i, name in enumerate(chosen_names)}

In [ ]:
##Team_Stats

def team_stats(team): #team is a list of 6 pokemon
    df = spark.sql(f"""
    SELECT name, Type1, Type2, attack, sp_attack, defense, sp_defense, hp, speed, battle_archetypes
    FROM df12mdp.brookefoye.pokemon_master""")
    df = df.filter(df['Name'].isin(team))
    return df

In [ ]:
##Unpack_Dict
def unpack(team):
    """Takes in a list of dicts, and returns a dict of each archytype and thier counts"""

    battle_count = {'juggernaut': 0, 'sweeper': 0, 'bulky_attacker': 0, 'glass_cannon': 0, 'support': 0, 'tank': 0, 'bulky_speedster': 0, 'attacker':0, 'slow_attacker':0, 'bulky': 0,'weak_slow':0, 'slow_support':0, 'fragile':0, 'balanced':0, 'midrange':0}
    for type in team:
        if type['battle_archetypes'] in battle_count:
            battle_count[type['battle_archetypes']] = type['count']
    return battle_count

In [ ]:
##Poketeam_Evaluator --done
"""Function that takes in a list of 6 pokemon, counts how many of each battle archytype is on that team, and returns an evaluation (string) stating if the team is: Over Powered (all beef), Achilies Heel (full of glass-cannons), Everyman(balanced), Underpowered (fragile), or Glass(weak)"""
def team_evaluator(team):
  result = ''
  full_team = team_stats(team)
  df = team_stats(team)
  df = df.groupBy('battle_archetypes').count()
  df = df.orderBy(df['count'].desc())
  #gets count of each battle archetype
  list_of_highest =[]
  list_of_lowest = []
  battle_list = [row.asDict() for row in df.collect()]
  count = unpack(battle_list)
  #gets a list of how many of each archetype there is in a dict
  highest = 0
  lowest = 6
  for counts in count: #cycle through dict to find highest count, lowest count, and a list of the top and bottom archtypes by count.
    if count[counts] == 0:
      continue
    if count[counts]>=highest:
      highest = count[counts]
      list_of_highest.append(counts)
    elif count[counts]<=lowest:
      lowest = count[counts]
      list_of_lowest.append(counts)
  #now compare the list of high and low archetypes to determine team type, use the top archytype to find the top pokemon on the team, and the weakest member(s)
  god_rank = ['juggernaut', 'sweeper', 'bulky_attacker'] 
  demigod_rank = ['glass_cannon', 'tank', 'support'] 
  hero_rank = ['bulky', 'bulky_speedster', 'attacker'] 
  mortal_rank = ['slow_attacker', 'slow_support', 'weak_slow']
  puny_rank = ['fragile', 'balanced', 'midrange']

  top = full_team.filter(full_team['battle_archetypes']==list_of_highest[0]).select('name').collect()[0]['name']
  try:
    bottom = full_team.filter(full_team['battle_archetypes']==list_of_lowest[-1]).select('name').collect()[0]['name']
  except IndexError:
      bottom = full_team.filter(full_team['battle_archetypes']==list_of_highest[-1]).select('name').collect()[0]['name']
  #RANKING
  if set(list_of_highest).issubset(god_rank):#this branch tests for mixing with god_tier
    if set(list_of_lowest).issubset(god_rank):#list is all god_lvl
        result = f'You have a team that is Powerful and Balanced! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(demigod_rank): 
        result = f'You have a team that is Legendary! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(hero_rank): 
        result = f'You have a team that is Epic! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(mortal_rank): 
        result = f'You have a team that is Uneven! Team MVP: {top}. Weakest Link: {bottom}. It is recomended that you switch out {bottom} with a pokemon that is more on par with the rest of your team.'
    elif set(list_of_lowest).issubset(puny_rank): 
        result = f'Your team has an Achilles Heel! Team MVP: {top}. Weakest Link: {bottom}. Please change out {bottom}, as they are too far below the level of {top}.'
    else:
        result = f'Your team is majority god tier! Team MVP: {top}. Weakest Link: {bottom}.'
  elif set(list_of_highest).issubset(demigod_rank):#this branch tests for mixing with demigod_rnk
    if set(list_of_lowest).issubset(demigod_rank):
        result = f'You have a team that is Powerful and Balanced! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(god_rank): 
        result = f'You have a team that is Legendary! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(hero_rank): 
        result = f'You have a team that is Powerful! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(mortal_rank): 
        result = f'You have a team that is Uneven! Team MVP: {top}. Weakest Link: {bottom}. It is recomended that you switch out {bottom} with a pokemon that is more on par with the rest of your team.'
    elif set(list_of_lowest).issubset(puny_rank): 
        result = f'Your team has an Achilles Heel! Team MVP: {top}. Weakest Link: {bottom}. Please change out {bottom}, as they are too far below the level of {top}.'
    else:
        result = f'Your team sure is really strong! Team MVP: {top}. Weakest Link: {bottom}.'
  elif set(list_of_highest).issubset(hero_rank):#this branch tests for mixing with hero_rank
    if set(list_of_lowest).issubset(god_rank):
        result = f'You have a team that is Epic! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(demigod_rank): 
        result = f'You have a team that is Powerful! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(hero_rank): 
        result = f'You have a team that is Strong and Balanced! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(mortal_rank): 
        result = f'You have a team that is Good! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(puny_rank): 
        result = f'You have a team that is Average! Team MVP: {top}. Weakest Link: {bottom}.'
    else:
        result = f'Your team is pretty strong! Team MVP: {top}. Weakest Link: {bottom}.'
  elif set(list_of_highest).issubset(mortal_rank):#this branch tests for mixing with mortal_rank
    if set(list_of_lowest).issubset(god_rank):
        result = f'You have a team that is Uneven! Team MVP: {top}. Weakest Link: {bottom}. Looks like most of your team is lower level than {top}, maybe save them as your Ace!'
    elif set(list_of_lowest).issubset(demigod_rank): 
        result = f'You have a team that is Average! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(hero_rank): 
        result = f'You have a team that is Good! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(mortal_rank): 
        result = f'You have a team that is Balanced! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(puny_rank): 
        result = f'You have a good starter team going! Team MVP: {top}. Weakest Link: {bottom}.'
    else:
        result = f'Your team sure is something! Team MVP: {top}. Weakest Link: {bottom}.'
  elif set(list_of_highest).issubset(puny_rank):#this branch tests for mixing with puny_rank
    if set(list_of_lowest).issubset(god_rank):
        result = f'You have a team that is Uneven! Team MVP: {top}. Weakest Link: {bottom}. Looks like most of your team is lower level than {top}, maybe save them as your Ace!'
    elif set(list_of_lowest).issubset(demigod_rank): 
        result = f'You have a team that is Uneven! Team MVP: {top}. Weakest Link: {bottom}. Looks like most of your team is lower level than {top}, maybe save them as your Ace!'
    elif set(list_of_lowest).issubset(hero_rank): 
        result = f'You have a team that is Average! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(mortal_rank): 
        result = f'You have a good starter team going! Team MVP: {top}. Weakest Link: {bottom}.'
    elif set(list_of_lowest).issubset(puny_rank): 
        result = f'You have a team that is Balanced! Team MVP: {top}. Weakest Link: {bottom}.'
    else:
        result = f'Your team is on the weaker side! Team MVP: {top}. Weakest Link: {bottom}.'
  else:
    result = f'Your team sure is a Mixed Bag! Team MVP: {top}. Weakest Link: {bottom}.'
  return result

In [ ]:
###Team_Generator --Works
from pyspark.sql import functions as F
def generate_team():
    team = False
    poke_types = ['ice', 'fire', 'water', 'grass', 'normal', 'electric', 'psychic', 'dark', 'fairy', 'fighting', 'rock', 'steel', 'bug', 'poison', 'flying', 'ghost', 'dragon', 'ground',]
    try:
        while team == False:
            gen = input("Hi! Please which generations you would like to use, seperated by spaces. Such as 1 2 5: ").strip() 
            gen = gen.split(" ")
            for i in gen:
                if i.isdigit() == False:
                    raise ValueError("Please enter a NUMBER.")
            i = int(i)
            if i>9 or i<1:
                    raise ValueError("Please enter a valid generation number between 1 and 9.")
    ##SUMMON JOINS FUNCTION
            team_source = join_gens(gen)
            team = True
        team = False 
        while team == False:
    ##After the main functionality works, add in a better allowance for duel types!!
    #types for real
            poke_type = input("Please enter some types you would like to use, separated by commas: ").strip()
            poke_type = [ptype.strip() for ptype in poke_type.lower().split(",")]
            for poke in poke_type:
                if poke not in poke_types:
                    raise ValueError("Sorry! You must enter valid types.")
            full_team = build_team(team_source, poke_type)
            pokemon_list = [value for value in full_team.values()]
            if len(pokemon_list) != 6:
                print("Sorry! There's been an error")
                break
            else:
                team = True
                print('Your team is ready!')
            pokemon_team = team_stats(pokemon_list)
            pokemon_team.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("df12mdp.brookefoye.pokemon_team")
            return (team_stats(pokemon_list))
    except ValueError as e:
        print(e) 
display(generate_team())

In [ ]:
###Pokemon_Switcher--Works
def poke_switch():
    from pyspark.sql import functions as F
    full_team = spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_team""")
    pokemon_list = full_team.select("Name").rdd.flatMap(lambda x: x).collect() #makes table column into a list
    ifThen = 1
    team = False
    try:
        while team == False:
            pokemon_switcher = input("Would you like to switch out a pokemon? (yes/no) ").strip()
            if pokemon_switcher.lower() == 'yes':
                pokemon_to_switch = input("Which pokemon would you like to switch out? ").strip()
            else:
                print("Aight, bye")
                ifThen = 0
                break
            if pokemon_to_switch not in pokemon_list:
                raise ValueError("Sorry! That pokemon isn't on your team.")
            else:
                new_pokemon = input("What pokemon would you like to switch to? ").strip()
                team_source = spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_master""")
                exists = team_source.filter(F.col("name") == new_pokemon).limit(1).count() > 0
                if not exists:
                    raise ValueError("Sorry! That pokemon isn't in the database.")
                else:
                    team = True
                    pokemon_list.remove(pokemon_to_switch)
                    pokemon_list.append(new_pokemon)
        #new pokemon in now in list of pokemon names
        if ifThen == 1:
            pokemon_team = team_stats(pokemon_list)
            pokemon_team.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("df12mdp.brookefoye.pokemon_team")
            return (team_stats(pokemon_list))
        else:
            pass
    except ValueError as e:
        print(e)

In [ ]:
###Build_Team from Nothing --Done
def poketeam_builder():
    """Based on pokemon name, enters pokemon and associated stats into a table, default to base stats"""
    slots = 0
    while slots == 0:
        chosen = input("Please enter 6 pokemon you would like for your team! You can seperate them by spaces, but spaces within names should be divided by (-) ")
        chosen = chosen.split(" ")
        for i in chosen:
            if i.isdigit() == True:
                raise ValueError("Please enter a valid pokemon name.")
            else:
                slots += 1
    team_source = spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_master""")
    for i in chosen:
        exists = team_source.filter(F.col("name") == i).limit(1).count() > 0
        if not exists:
            raise ValueError("Sorry! That pokemon isn't in the database.")
        else:
            team = True
            print("Your team is ready!")
            break
    pokemon_team = team_stats(chosen)
    pokemon_team.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("df12mdp.brookefoye.pokemon_team")
    return (team_stats(chosen))
display(poketeam_builder())

In [ ]:
###Calc_Archetype --Works-ish
def calc_arch(pokemon): 
    """Calculates a fresh archetype for an individual pokemon based on the name. It takes the current stats and compares them to the bottom 33%, middling 67% and the top percent of the database's sums for each column."""
    from pyspark.sql import functions as F    
    #Calculations >33 is low, 33-67 is mid, 67+ is high
    full_list = spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_master""")
    #loading base sums
    bottom33_bulk = full_list.approxQuantile('bulk', [0.30], 0.01)[0]
    mid33_bulk = full_list.approxQuantile('bulk', [0.67], 0.01)[0]
    
    bottom33_offense = full_list.approxQuantile('offense', [0.05], 0.01)[0]
    mid33_offense = full_list.approxQuantile('offense', [0.67], 0.01)[0]

    bottom33_speed = full_list.approxQuantile('speed', [0.05], 0.01)[0]
    mid33_speed = full_list.approxQuantile('speed', [0.67], 0.01)[0]

    #get pokemon's current stats
    team_list = spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_team""")
    current_speed = team_list.filter(F.col("name") == pokemon).select("speed").collect()[0]["speed"]

    current_attack = team_list.filter(F.col("name") == pokemon).select("attack").collect()[0]["attack"]
    current_sa = team_list.filter(F.col("name") == pokemon).select("sp_attack").collect()[0]["sp_attack"]
    #calc current offense
    current_defense = team_list.filter(F.col("name") == pokemon).select("defense").collect()[0]["defense"]
    current_spd = team_list.filter(F.col("name") == pokemon).select("sp_defense").collect()[0]["sp_defense"]
    current_hp = team_list.filter(F.col("name") == pokemon).select("hp").collect()[0]["hp"]
    #calc current bulk
    current_bulk = current_hp + current_defense + current_spd
    current_offense = current_attack + current_sa
    #Calculate Archetypes

    archType = ""

    if current_offense > mid33_offense: #High Offense
        if current_bulk > mid33_bulk:
            if current_speed > mid33_speed:
                archType = 'juggernaut'
            elif current_speed < mid33_speed and current_speed > bottom33_speed:
                archType = 'bulky_attacker'
        elif current_bulk <mid33_bulk and current_bulk > bottom33_bulk: #High offense, mid bulk
            if current_speed >mid33_speed: #high speed
                archType = "sweeper"
            elif current_speed > bottom33_speed and current_speed <mid33_speed: #mid speed
                archType = "attacker"
        elif current_bulk <bottom33_bulk:
            if current_speed > bottom33_speed: #mid/high speed
                archType = "glass_cannon"

    elif current_offense > bottom33_offense: #mid to high offense
        if current_bulk > bottom33_bulk and current_speed < bottom33_speed:
            archType = "slow_attacker" #corrected, might change places
        if current_bulk >mid33_bulk: #High bulk
            if current_speed > mid33_speed:
                archType = 'bulky_speedster'
            elif current_speed < mid33_speed and current_speed > bottom33_speed:
                archType = "bulky"
        elif current_bulk < mid33_bulk and current_bulk > bottom33_bulk: #all mids
            if current_speed < mid33_speed and current_speed > bottom33_speed:
                archType = "balanced"
    elif current_offense < bottom33_offense:  #Low offense
        if current_bulk >mid33_bulk: #High bulk
            if current_speed > bottom33_speed:
                archType = "support"
            elif current_speed < bottom33_speed:
                archType = "tank"
        elif current_bulk > bottom33_bulk: #mid/high bulk
            if current_speed < bottom33_speed:
                archType = "slow_support"
        elif current_bulk < bottom33_bulk: #low bulk and offense
            if current_speed > bottom33_speed:
                archType = "fragile"
            elif current_speed < mid33_speed: #low everything
                archType = "weak_slow"
    return archType

#special_defense = special attacks/magic
#hp + defense + special_defense = bulk
#attack + special attack = offense, intigrate??

In [ ]:
###Team Editor
def team_edit():
    from pyspark.sql import functions as F
    """This function takes the df12mdp.brookefoye.pokemon_team table and allows the user to edit individual stats for a pokemon, altering the battle archetype automatically"""
    source_list = spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_team""")
    pokemon = input("Hello! Please enter the name of the pokemon you wish to edit: ")
    if pokemon not in source_list.select("Name").rdd.flatMap(lambda x: x).collect():
        raise ValueError("Sorry! That pokemon isn't on your team.")
    else:
        stat = input("Please enter the stat you wish to edit: ")
        if stat not in source_list.columns:
            raise ValueError("Sorry! That stat isn't in the database.")
        else:
            value = input("Please enter the value you wish to change it to: ")
            value = int(value)
            source_list = source_list.withColumn(stat, F.when(F.col("Name") == pokemon, value).otherwise(F.col(stat)))
            source_list.write.mode("overwrite").saveAsTable("df12mdp.brookefoye.pokemon_team")
    #now edit battle archetype
    new_type = calc_arch(pokemon)
    source_list = source_list.withColumn('battle_archetypes', F.when(F.col("Name") == pokemon, new_type).otherwise(F.col('battle_archetypes')))
    return source_list

display(spark.sql("""SELECT * FROM df12mdp.brookefoye.pokemon_team"""))#control
display(team_edit())